In [ ]:
from pathlib import Path
from eduray import *

# All rendered images from this notebook are saved into this directory.
OUTPUT_DIR = Path("images")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Demo scene setup and example of using the ray tracer

At the end of this series of notebooks, we create a more complex scene that demonstrates the capabilities of the ray tracer. The scene includes multiple objects, different materials, several light sources, and a composed camera view.

The goal is to showcase features implemented in the previous notebooks, such as reflections, refractions, shadows, procedural textures, and recursive ray tracing. If you are starting here without going through the previous notebooks, this final scene still gives a compact overview of what the ray tracer can do and how a complete scene can be set up.

---
### Colors

Colors can be defined in several ways:

- `Color.custom_rgb(r, g, b)` uses common 8-bit RGB values in the range `0–255`.
- `Color.linear_rgb(r, g, b)` uses linear RGB values in the range `0.0–1.0`.
- `Color(r, g, b)` can also be used directly for linear RGB values, but the `Color.linear_rgb(...)` method is more explicit and helps avoid confusion about the expected value range.

For most notebook examples, `Color.custom_rgb(...)` is the easiest option because it uses the same value range as common color pickers.

In [ ]:
white = Color.custom_rgb(255, 255, 255)
blue = Color.linear_rgb(0.118, 0.565, 1.0)
green = Color(0.133, 0.694, 0.298)

---
### Materials

This section defines the materials used in the demo scene.
Each material controls how an object reacts to light. The most important parameters are:

- `base_color` — the main diffuse color of the object,
- `spec_color` — the color of the specular highlight,
- `shininess` — the size and sharpness of the highlight,
- `reflectivity` — how strongly the material reflects the scene,
- `transparency` — how much light passes through the material,
- `ior` — the index of refraction used for transparent objects,
- `normal_noise` or `color_noise` — procedural variation of the surface.

The scene combines several material types: diffuse, shiny, reflective, transparent, marble-like, and procedurally perturbed materials. This makes it possible to demonstrate local lighting, reflections, refractions, and procedural surface detail in one scene.

> Try it yourself: change the color, shininess, reflectivity, or transparency of one material and render the scene again. For example, increase the `reflectivity` of a sphere, lower the `shininess`, or remove `normal_noise` to see how the surface changes.

> Tip: Find material IOR values for common materials online to create more realistic renderings. For example, glass typically has an IOR around 1.5, while diamond is around 2.42. Adjusting the IOR can significantly affect how light bends and reflects on transparent objects.

In [ ]:
# REFLECTIVE MATERIALS
right_mirror = PhongMaterial(
    name="mirror",
    base_color=Color.custom_rgb(200, 200, 200),
    spec_color=white,
    shininess=500, # very high shininess for a perfect mirror-like surface
    reflectivity=0.95, # very high reflectivity for an almost mirror-like surface
)

left_tinted_mirror = PhongMaterial(
    name="tinted_mirror",
    base_color=Color.custom_rgb(200, 220, 255),
    spec_color=white,
    shininess=500,
    reflectivity=0.8, # slightly lower reflectivity to allow more of the tint color to show through in reflections, creating a subtle blue hue in reflected objects
)

# bumpy materials
bumpy_blue = PhongMaterial(
    name="bumpy_blue",
    base_color=blue,
    spec_color=white,
    shininess=50,
    reflectivity=0.2,
    transparency=0.0,
    normal_noise=PerlinNoise(
        scale=3.5,
        strength=0.1
    ),
)

bumpy_green = PhongMaterial(
    name="bumpy_green",
    base_color=green,
    spec_color=white,
    shininess=50,
    reflectivity=0.2,
    transparency=0.0,
    normal_noise=PerlinNoise(
        scale=3.5,
        strength=0.06
    ),
)

bumpy_red = PhongMaterial(
    name="bumpy_red",
    base_color=Color.custom_rgb(255, 10, 40),
    spec_color=white,
    shininess=50,
    reflectivity=0.2,
    transparency=0.0,
    normal_noise=PerlinNoise(
        scale=3.5,
        strength=0.06
    ),
)

golden = PhongMaterial(
    name="golden",
    base_color=Color.custom_rgb(255, 215, 0),
    spec_color=white,
    shininess=100, # moderate shininess for a slightly rough gold look
    reflectivity=0.8, # high reflectivity for a metallic surface
    transparency=0.0,
)

# TRANSPARENT MATERIALS
diamond = PhongMaterial(
    name="diamond",
    base_color=Color.custom_rgb(210, 245, 255),
    spec_color=white,
    shininess=250, # very high shininess for sharp highlights
    ior=2.42, # typical IOR for diamond
    transparency=0.92, # high transparency to allow light to pass through, but not 100% to create a more realistic look
    normal_noise=TurbulenceNoise( # noise to see how it affects refractions and reflections, creating a more interesting and less perfect surface.
        scale=2.0,
        strength=0.1,
        octaves=4,
        lacunarity=2.0,
        gain=0.5
    ),
)

crystal_no_noise = PhongMaterial( # similar to diamond but without noise for a cleaner look
    base_color=Color.custom_rgb(210, 245, 255),
    spec_color=white,
    shininess=250,
    ior=2.0, # slightly lower IOR for a less extreme refraction than diamond
    reflectivity=0.08,
    transparency=0.92,
)


# PROCEDURAL MATERIALS
marble = MarbleMaterial(
    name="marble",
    base_color=Color.custom_rgb(220, 220, 220),
    spec_color=white,
    color_one_factor= 0.1,
    color_two_factor= 1.0,
    shininess=20,
    ior=1.5,
    reflectivity=0.1,
    transparency=0.0,
    normal_noise=TurbulenceNoise( # small turbulence noise to create subtle surface variations and enhance the marble texture
        scale=5.0, # higher scale means higher spatial frequency, so the noise features become smaller
        strength=0.1, # low strength to create gentle surface perturbations without overwhelming the base marble pattern
        octaves=4, # number of noise layers to combine, more octaves create more complex and detailed noise patterns
        lacunarity=2.0, # frequency multiplier for each octave, controls how quickly the noise frequency increases with each layer
        gain=0.5 # amplitude multiplier for each octave, controls how quickly the noise amplitude decreases with each layer
    ),
)

# try it yourself - use this material on some objects to see how it looks. Also try to change parameters
rock = RockMaterial(
        base_color=Color.custom_rgb(110, 90, 90),
        spec_color=Color.custom_rgb(160, 110, 110),
        ambient_color=Color.custom_rgb(200, 190, 190),
        shininess=8,
        color_noise=FBMNoise( # fractal brownian motion noise to create complex color variations and enhance the rock appearance
            scale=1.2,
            strength=2.0,
            octaves=7,
            lacunarity=2.0,
            gain=0.5,
        ),
        normal_noise=TurbulenceNoise( # turbulence noise to create surface perturbations and enhance the rough rock texture
            scale=2.0,
            strength=0.5,
            octaves=5,
            lacunarity=2.0,
            gain=0.5,
        ),
        color_scale=3.0,
    )

---
### Objects and geometry
This section defines the objects in the scene, their geometry, and the materials applied to them. The scene includes a ground plane, a central pedestal, a diamond sphere as the hero object, floating rings around the hero, additional colored spheres for visual interest, and mirror panels on the sides to create interesting reflections. Each object is created with a specific geometry (e.g., `Sphere`, `Box`, `Torus`, `Square`) and then transformed (scaled, rotated, translated) to achieve the desired composition in the scene.

> **Try it yourself:** transform objects by changing their scale, rotation, or translation parameters. For example, you can make the diamond sphere larger or smaller, rotate the rings differently, or move the colored spheres to new positions. Experimenting with these transformations can help you understand how object placement and orientation affect the final rendered image.

In [ ]:
objects=[
    # GROUND PLANE
    Object(
        geometry=Square(
            vertex=Vertex(-1, 0, -1),
            diagonal_vertex=Vertex(1, 0, 1),
        ),
        material=marble,
    ).scale(8.0, 1.0, 10.0).translate(0, -1.0, -6.0),

    # CENTRAL PEDESTAL
    Object(
        geometry=Box(
            corner1=Vertex(-0.5, -0.5, -0.5),
            corner2=Vertex(0.5, 0.5, 0.5),
        ),
        material= golden,
    ).scale(1.7, 0.5, 1.7).translate(0.0, -0.75, -6.0),

    # HERO OBJECT - DIAMOND SPHERE
    Object(
        geometry=Sphere(
            center=Vertex(0, 0, 0),
            radius=1.0,
        ),
        material=diamond,
    ).scale(1.0, 1.0, 1.0).translate(0.0, 0.7, -6.0),

    # FLOATING RING AROUND HERO
    Object(
        geometry=Torus(
            center=Vertex(0, 0, 0),
            radius_major=1.5,
            radius_tube=0.12,
        ),
        material=golden,
    ).rotate_x(72).rotate_y(22).translate(0.0, 0.9, -6.0),

    # SECOND RING
    Object(
        geometry=Torus(
            center=Vertex(0, 0, 0),
            radius_major=1.2,
            radius_tube=0.08,
        ),
        material=left_tinted_mirror,
    ).rotate_y(90).rotate_z(30).translate(0.0, 0.9, -6.0),

    # right blue bumpy sphere
    Object(
        geometry=Sphere(
            center=Vertex(0, 0, 0),
            radius=1.0,
        ),
        material=bumpy_blue,
    ).scale(0.8,0.8,0.8).translate(2.2, -0.2, -4.8),

    # left red bumpy sphere
    Object(
        geometry=Sphere(
            center=Vertex(0, 0, 0),
            radius=1.0,
        ),
        material=bumpy_red,
    ).scale(0.8,0.8,0.8).translate(-2.2, -0.2, -4.8),

    # middle green sphere
    Object(
        geometry=Sphere(
            center=Vertex(0, 0, 0),
            radius=1.0,
        ),
        material=bumpy_green,
    ).scale(0.8,0.8,0.8).translate(0.0, -0.2, -8.5),

    # left mirror panel
    Object(
        geometry=Square(
            vertex=Vertex(-1, 0, -1),
            diagonal_vertex=Vertex(1, 0, 1),
        ),
        material=left_tinted_mirror,
    ).scale(1.5, 1.0, 8.5).rotate_x(90).rotate_y(38).translate(-4.6, 1.0, -10.0),

    # right mirror panel
    Object(
        geometry=Square(
            vertex=Vertex(-1, 0, -1),
            diagonal_vertex=Vertex(1, 0, 1),
        ),
        material=right_mirror,
    ).scale(1.5, 1.0, 8.5).rotate_x(90).rotate_y(-38).translate(4.6, 1.0, -10.0),

    # small crystal sphere above the diamond to visualize refraction and reflections in multiple refractive objects, including noise one
    Object(
        geometry=Sphere(
            center=Vertex(0, 0, 0),
            radius=1.0,
        ),
        material=crystal_no_noise,
    ).scale(0.4, 0.4, 0.4).translate(0.0, 1.5, -2.0),
]

---
### Camera
The camera is set up as a `PinholeCamera` with a specific field of view, position, and direction. The camera's `origin` is placed slightly above the ground and a bit back from the scene to capture a good view of the objects. The `direction` is set to look straight ahead along the negative Z-axis, which is the default forward direction in this coordinate system. The `fov_deg` parameter controls how wide the camera's field of view is, affecting how much of the scene is visible and how objects are projected onto the image plane.

> **Try it yourself:** Adjust the camera's position, direction, or field of view to see how it changes the composition of the rendered image.

In [ ]:
camera=(
    PinholeCamera(
    fov_deg=37,
    origin=Vertex(0.0, 0.8, 2.0),
    direction=Vector(0.0, 0, -1.0)
    )
)

---
### Lights
Three point lights and one ambient light are used to illuminate the scene. The point lights are strategically placed to create a visually appealing composition with a warm key light to see how shadows are formed, a cool fill light to soften shadows and add color contrast, and a pink rim light to create highlights and accentuate the edges of objects. The ambient light provides a base level of illumination to ensure that no part of the scene is completely dark.

> **Try it yourself:** Change the position, intensity, or color of the lights to see how it affects the mood and appearance of the scene. Or use Lights with different types, such as `DirectionalLight`, `SpotLight` or lights with falloff to create different lighting effects.

In [ ]:
lights=[
    # main warm key light
    PointLight(
        position=Vertex(3.5, 5.5, -1.0),
        intensity=1.0,
        color=Color.custom_rgb(255, 244, 230),
    ),
    # cool fill light
    PointLight(
        position=Vertex(-4.5, 2.8, -2.5),
        intensity=0.45,
        color=Color.custom_rgb(200, 220, 255),
    ),
    # pink rim / accent light
    PointLight(
        position=Vertex(0.0, 4.0, -10.5),
        intensity=0.6,
        color=Color.custom_rgb(255, 180, 220),
    ),
    AmbientLight(
        intensity=0.2,
        color=Color.custom_rgb(255, 255, 255),
    )
]

---
### Render configuration and preview settings
The `RenderConfig` is set to low resolution for faster experimentation, with a small number of samples per pixel and a low maximum depth to speed up rendering while still demonstrating the scene's features. The `PostProcessConfig` is disabled for this demo to show the raw output of the ray tracer without any upscaling or additional processing.

> **Try it yourself:**  Change depth and samples per pixel to see how it affects the quality and rendering time.

#### Approximate recursive ray count

The number of rays grows quickly with recursion depth. In the worst case, each hit can spawn both a reflection ray and a refraction ray. Ignoring shadow rays, this forms a binary tree of rays.

The approximate number of recursive rays per sample is:

$$
1 + 2 + 4 + \dots + 2^d = 2^{d+1} - 1
$$

where $ d $ is `max_depth`.

| Max Depth | Rays per Sample | SPP = 1 | SPP = 2 | SPP = 4 | SPP = 8 |
|-----------|-----------------|---------|---------|---------|---------|
| 0         | 1               | 1       | 2       | 4       | 8       |
| 1         | 3               | 3       | 6       | 12      | 24      |
| 2         | 7               | 7       | 14      | 28      | 56      |
| 3         | 15              | 15      | 30      | 60      | 120     |
| 4         | 31              | 31      | 62      | 124     | 248     |
| 5         | 63              | 63      | 126     | 252     | 504     |
| 6         | 127             | 127     | 254     | 508     | 1016    |
| 7         | 255             | 255     | 510     | 1020    | 2040    |
| 8         | 511             | 511     | 1022    | 2044    | 4088    |

These values are worst-case estimates for primary, reflection, and refraction rays only. The real number is often way lower because not every ray hits a transparent surface every time. However, the render can still be slow because visible hits may also cast shadow rays toward the lights, and each ray must be tested against the scene geometry.

For a higher-quality setting such as `Resolution.R480p`, `samples_per_pixel=4`, and `max_depth=4`, the renderer may cast up to approximately:

`854 × 480 × 124 ≈ 50.8 million recursive rays`

in the worst case, ignoring additional shadow rays. Each ray must be tested against the scene geometry. Since this educational renderer does not yet use an acceleration structure such as BVH, the cost grows with the number of objects. In addition, visible shaded hits may cast shadow rays toward the lights, which further increases render time.

This is one reason why the final demo scene can be much slower than the earlier simple examples.

In [ ]:
render_configuration = RenderConfig(
    resolution=Resolution.R144p,
    samples_per_pixel=1,
    max_depth=5,
)

post_process_config = PostProcessConfig(
    enabled=False,
    scale_factor=2,
)

preview_config = PreviewConfig(
    progress_display = ProgressDisplay.TQDM_IMAGE_PREVIEW,
)

---
### Scene assembly and rendering
Finally, we assemble the final scene

> **Try it yourself:** Download HDR skybox and test it in the scene. The skybox files are not included in the repository because they can be large. You can find free HDR skyboxes on websites such as [Poly Haven](https://hdrihaven.com/), often in very high resolutions. Download, uncomment the `skybox` parameter in the `Scene` definition, and provide the path to your downloaded HDR file. You should see the skybox color appearing in the background and in reflections and refractions, adding more visual interest to the scene and demonstrating how the integrator handles rays that miss all geometry.

In [ ]:
scene = Scene(
    camera=camera,
    objects=objects,
    lights=lights,
    #skybox="./skyboxes/sand.hdr",
)
scene.validate()

# Final rendering

In [ ]:
my_ray_tracer = LinearRenderLoop(scene=scene, render_config=render_configuration, preview_config=preview_config, post_process_config=post_process_config)
png_path = my_ray_tracer.render("images/demo.png")
ipynb_display_images(png_path)

---
## Final thoughts

This notebook concludes the educational series on building a ray tracer from scratch.

Across the series, we moved from basic image and ray representation to cameras, intersections, surface normals, local lighting, shadows, reflections, refractions, procedural materials, and finally a complete demo scene.

The final scene combines these parts into one example and demonstrates how materials, lights, geometry, recursive rays, and procedural effects influence the rendered image.

Try experimenting with the scene by changing materials, lights, objects, or rendering settings. You can also return to earlier notebooks and implement your own methods or extend the ray tracer with new features.

Happy ray tracing!
